In [1]:
import os, sys
import numpy as np
import pandas as pd
import re
import random
from datetime import date
import time
from collections import Counter
from sklearn.model_selection import train_test_split
parts=os.getcwd().split('/')[:-1]

code_path='/'.join(parts)+'/scripts'
sys.path.append(code_path)
from experiment_utils import data_normalization

main_data_dir='/'.join(os.getcwd().split('/')[:-3])
datapath='/'.join(parts)+'/data/'
sys.path.append(datapath)
%load_ext autoreload

/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq


### CDC Diabetes dataset

In [2]:
diabetes_datapath=main_data_dir+'/datasets/cdc_diabetes/'
diabetes_detail=pd.read_csv(diabetes_datapath+'diabetes_012_health_indicators_BRFSS2015.csv')
features=diabetes_detail.columns[1:]
diabetes_detail.head(4)

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0


In [3]:
print(diabetes_detail['Diabetes_012'].value_counts())

Y=diabetes_detail[diabetes_detail.columns[0]]#.copy()
X=diabetes_detail[features].copy()
print(len(Y), X.shape)
print(Counter(Y))
X_train, X_test12, Y_train, Y_test12 = train_test_split(X, Y, test_size=0.40, random_state=42)
#X_test1, X_test2, Y_test1, Y_test2 = train_test_split(X_test12, Y_test12, test_size=0.50, random_state=42)
print('Train:\n',Counter(Y_train))
print('Test-1:\n',Counter(Y_test12))
#print('Test-2:\n',Counter(Y_test1))
trainset, testset=X_train.copy(),X_test12.copy()
trainset['Label'], testset['Label']=Y_train,Y_test12

dname='diabetes'
if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
    print('Files do not exist')
    trainset.to_csv(datapath+'%s/%s_trainset2.csv'%(dname,dname), index=False)
    testset.to_csv(datapath+'%s/%s_testset2.csv'%(dname,dname), index=False)
else:
    print('Files exist')

zXtrain, zXtest=data_normalization(X_train, X_test12)

Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64
253680 (253680, 21)
Counter({0.0: 213703, 2.0: 35346, 1.0: 4631})
Train:
 Counter({0.0: 128134, 2.0: 21308, 1.0: 2766})
Test-1:
 Counter({0.0: 85569, 2.0: 14038, 1.0: 1865})
Files exist


### Diabetes dataset 130 US Hospitals

In [2]:
#!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo 
# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
diabetes_df= diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
print(diabetes_df.shape)
X= diabetes_130_us_hospitals_for_years_1999_2008.data.features 

print('Shape of diabetes_df', diabetes_df.shape)
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
label_dict={'NO':0, '>30':1, '<30':1}
diabetes_df['Label']=y
diabetes_df['Label']=diabetes_df['Label'].map(label_dict)
X=diabetes_df.copy()

(101766, 47)
Shape of diabetes_df (101766, 47)


/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [3]:
diabetes_df=X.copy()
# remove select columns
remove_cols = ['weight','diag_1', 'diag_2', 'diag_3','payer_code', 'admission_type_id', 'admission_source_id',
              'medical_specialty']
if len(remove_cols) > 0:
    diabetes_df= diabetes_df.drop(columns=remove_cols)
##############################################################
print(diabetes_df['A1Cresult'].value_counts())
disch_dict={'Disch_Home/hospice':[1,6,8, 12,13, 14], 'Expired': [11,19,20,21],
            'Disch_Left AMA': [7],   'Disch_Unknown': [18,25, 26], 
            'Disch_Elsewhere': [2,3,4,5,9,10,15,16,17,22,23,24,27,28, 29, 30],}
diabetes_df['discharge_summary']=''
disch_col='discharge_disposition_id'
diabetes_df[disch_col]=diabetes_df[disch_col].astype(int)
for key, val in disch_dict.items():
    diabetes_df.loc[diabetes_df[disch_col].isin(val),'discharge_summary']=key

encoded_df1 = pd.get_dummies(diabetes_df['discharge_summary'], columns=['discharge_summary'], dtype=float)
binary_cats1=list(encoded_df1.columns)
encoded_df2 = pd.get_dummies(diabetes_df['race'], columns=['race'], dtype=float)
binary_cats2=list(encoded_df2.columns)
diabetes_df=pd.concat([diabetes_df, encoded_df1, encoded_df2], axis=1)
diabetes_df['Race_other']=diabetes_df[['Asian', 'Hispanic', 'Other']].sum(axis=1)#>0.any()
diabetes_df['Race_other'].loc[diabetes_df['Race_other']>0]=1
diabetes_df.drop(['discharge_summary', 'discharge_disposition_id', 'race', 'Asian', 'Hispanic', 'Other'],
                 axis=1, inplace=True)
binary_cats2=list(np.union1d(np.setdiff1d(binary_cats2,[ 'Asian', 'Hispanic', 'Other']),['Race_other']))
#################################################################################################################
################################################ Originally numeric #############################################
numeric = ['time_in_hospital', 'num_lab_procedures','num_procedures', 'num_medications','number_outpatient', 
           'number_emergency', 'number_inpatient','number_diagnoses']
##################################################################################################################
################################################### Age ##########################################################
age_count, age_val=diabetes_df['age'].value_counts(),[]
for s in age_count.keys():
    matches = re.findall(r'-?\d*\.?\d+', s)
    res = [float(x) if '.' in x else int(x) for x in matches]
    age_val.append((np.abs(res[1])+res[0])/2)
age_dict=dict(zip(age_count.keys(), age_val))

A1Cresult
>8      8216
Norm    4990
>7      3812
Name: count, dtype: int64


/tmp/ipykernel_94300/813194512.py:24: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  diabetes_df['Race_other'].loc[diabetes_df['Race_other']>0]=1


In [63]:
binary_cats2

['AfricanAmerican', 'Caucasian', 'Race_other']

In [4]:
############################################################################################
###################################### Dosage categories ###################################
med_cats=['A1Cresult','acetohexamide','acarbose', 'citoglipton', 'chlorpropamide', 'examide', 'insulin',
      'glimepiride','glyburide-metformin','glimepiride-pioglitazone','glipizide', 'glipizide-metformin', 'glyburide', 
       'metformin-pioglitazone', 'miglitol','metformin-rosiglitazone', 'metformin', 'nateglinide',
       'pioglitazone', 'repaglinide','rosiglitazone','max_glu_serum', 'tolazamide', 'tolbutamide', 'troglitazone']
cats,cat2num=med_cats+['age'],[]
dosage_dict={'Up':1,'Steady':0.5, 'Down':0.25, 'No':0,
            'Norm':0,'>7':1,'>8':2, '>200':1,'>300':2}
for c in cats:
    if c in med_cats:
        diabetes_df['num_%s'%c]=diabetes_df[c].map(dosage_dict)
        cat2num.append('num_%s'%c)
    else:
        if c=='age':
            diabetes_df['num_cat_%s'%c]=diabetes_df[c].map(age_dict)
        else:
            print('Empty')
        cat2num.append('num_cat_%s'%c)
    del diabetes_df[c]
###################################################################################################################
#################################################### Binary vars ##################################################
bin_vars,bin2num=['gender', 'change', 'diabetesMed'],[]
for c in bin_vars:
    print(diabetes_df[c].value_counts())
binary_dict={'Yes':1, 'No':0,'unknown':-1, 'Male':1, 'Female':0, 'Unknown/Invalid':-1, 'Ch':1}
for b in bin_vars:
    diabetes_df['num_%s'%b]=diabetes_df[b].map(binary_dict)
    bin2num.append('num_%s'%b)
    del diabetes_df[b]
####################################################################################################################
############################@################## Missing data per category ##########################################
numeric_all=bin2num+numeric+cat2num+binary_cats1+binary_cats2
missing_per_col=diabetes_df[numeric_all].isnull().mean()
remove_nan_cols=missing_per_col.keys()[missing_per_col.values>0.3]
print('Cols to be removed for lack of enough data', remove_nan_cols)
if len(remove_nan_cols) > 0:
    diabetes_df= diabetes_df.drop(columns=remove_nan_cols)
# remove_singular cols
#diabetes_df=diabetes_df[diabetes_df.columns[diabetes_df.std()>0]]
###################################################################################################################
cat2num=list(np.setdiff1d(cat2num, remove_nan_cols))
numeric_all=bin2num+numeric+cat2num+binary_cats1+binary_cats2
print('Unaccounted columns are: ', np.setdiff1d(diabetes_df.columns, numeric_all))
print(np.setdiff1d(diabetes_df.columns, numeric_all))
####################################################################################################################
############################################## split into train and test ###########################################
diabetes_df=diabetes_df[numeric_all+['Label']].copy()
Xtrain, Xtest, Ytrain, Ytest = train_test_split(diabetes_df[numeric_all], diabetes_df['Label'], test_size=0.20, random_state=42)
trainset, testset=Xtrain.copy(),Xtest.copy()
trainset['Label'],testset['Label']=Ytrain, Ytest
dname='diabetes'
if os.path.exists(datapath+'%s/%s_trainset1.csv'%(dname,dname))==False:
    print('Files do not exist')
    trainset.to_csv(datapath+'%s/%s_trainset.csv'%(dname,dname), index=False)
    testset.to_csv(datapath+'%s/%s_testset.csv'%(dname,dname), index=False)
else:
    print('Files exist')
    trainset=pd.read_csv(datapath+'%s/%s_trainset.csv'%(dname,dname))
    testset=pd.read_csv(datapath+'%s/%s_testset.csv'%(dname,dname))

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64
change
No    54755
Ch    47011
Name: count, dtype: int64
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64
Cols to be removed for lack of enough data Index(['num_A1Cresult', 'num_max_glu_serum'], dtype='str')
Unaccounted columns are:  ['Label']
['Label']
Files do not exist


In [81]:
current_meds=np.setdiff1d(np.intersect1d(cat2num, list(diabetes_df.columns)),['num_cat_age'])
print(current_meds)
mets=['num_metformin','num_rosiglitazone', 'num_metformin-rosiglitazone']
diabetes_df['mets']=diabetes_df[mets].sum(axis=1)
print(np.sum(diabetes_df['mets']==0.75))
diabetes_df[mets+['mets']]

['num_acarbose' 'num_acetohexamide' 'num_chlorpropamide' 'num_citoglipton'
 'num_examide' 'num_glimepiride' 'num_glimepiride-pioglitazone'
 'num_glipizide' 'num_glipizide-metformin' 'num_glyburide'
 'num_glyburide-metformin' 'num_insulin' 'num_metformin'
 'num_metformin-pioglitazone' 'num_metformin-rosiglitazone' 'num_miglitol'
 'num_nateglinide' 'num_pioglitazone' 'num_repaglinide'
 'num_rosiglitazone' 'num_tolazamide' 'num_tolbutamide' 'num_troglitazone']
90


,num_metformin,num_rosiglitazone,num_metformin-rosiglitazone,mets
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0
...,...,...,...,...
101761,0.5,0.0,0.0,0.5
101762,0.0,0.0,0.0,0.0
101763,0.5,0.0,0.0,0.5
101764,0.0,0.0,0.0,0.0


### Criteo uplift dataset


In [3]:
critero_datapath=main_data_dir+'/datasets/ci_ad_effect/'
dname='criteo'
if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
    print('File do not exist yet')
    criteo_train_df=pd.read_csv(critero_datapath+'train/uplift1_consolidated.csv.gzip', compression='gzip')
    criteo_val_df=pd.read_csv(critero_datapath+'train/uplift2_consolidated.csv.gzip', compression='gzip')
    criteo_test_df=pd.read_csv(critero_datapath+'test/uplift_test_consolidated.csv.gzip', compression='gzip')
    ############################################################################################################
    criteo_trainset=pd.concat([criteo_train_df, criteo_test_df], axis=0)
    criteo_trainset=criteo_trainset[criteo_trainset['visit'].astype(int)<2].copy()
    criteo_val_df=criteo_val_df[criteo_val_df['visit'].astype(int)<2].copy()
    criteo_trainset.to_csv('%s%s/%s_trainset.csv'%(datapath,dname,dname), index=False)
    criteo_val_df.to_csv('%s%s/%s_testset.csv'%(datapath,dname,dname), index=False)
    print(criteo_trainset['subset'].value_counts())
    print(criteo_val_df['subset'].value_counts())
else:
    print('Files exist')
    criteo_trainset=pd.read_csv('%s%s/%s_trainset.csv'%(datapath,dname,dname))
    #criteo_val_df=pd.read_csv('%s%s/%s_valset.csv'%(datapath,dname,dname))
    criteo_testset=pd.read_csv('%s%s/%s_testset.csv'%(datapath,dname,dname))

total_df_size=criteo_trainset.shape[0]+criteo_testset.shape[0]
print(criteo_trainset.shape[0]/total_df_size, criteo_testset.shape[0]/total_df_size)
print(criteo_train_df.shape[0]/total_df_size, criteo_val_df.shape[0]/total_df_size, criteo_test_df.shape[0]/total_df_size)

criteo_train_df.head(4)

Files exist
0.8199998841167814 0.18000011588321863
0.4199998411970708 0.18000011588321863 0.4000001144525616


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,subset,exposure,shuff_group
0,12.616365,10.059654,9.006692,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,0,0,0,S0,0,38
1,23.370856,10.059654,8.214383,4.679882,10.280525,4.115453,-8.493011,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,S0,0,16
2,24.235198,10.059654,8.214383,4.679882,10.280525,4.115453,-5.116672,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,S0,0,6
3,24.846428,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,S0,0,46


In [4]:
criteo_trainset['visit'].value_counts()

visit
0    10924582
1      538681
Name: count, dtype: int64

In [17]:
#data[(data.marks != 98)
#.copy
criteo_trainset['visit'].value_counts()

visit
0    10924582
1      538681
Name: count, dtype: int64

## Banking dataset

In [34]:

banking_datapath=main_data_dir+'/datasets/banking/'

dname='banking'
if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
    print('Files do not exist')
    banking_df = pd.read_csv(banking_datapath+'bank-additional-full.csv', sep=';')
    # remove select columns
    remove_cols = []
    if len(remove_cols) > 0:
        banking_df = banking_df.drop(columns=remove_cols)
    # remove nan rows
    nan_rows = banking_df[banking_df.isnull().any(axis=1)]
    print('nan rows: {}'.format(len(nan_rows)))
    banking_df = banking_df.dropna()
    columns = list(banking_df.columns)
    Label = ['y']
    numeric = ['age', 'duration', 'campaign', 'pdays', 'previous','emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
               'euribor3m', 'nr.employed']
    categorical=['job', 'marital','education', 'housing','loan', 'contact', 'default', 'poutcome']
    features=numeric+categorical+Label
    trainset=banking_df[features].copy()
    edu_dict={'illiterate':-1, 'unknown':0, 'basic.4y':1, 'basic.6y':2, 'basic.9y':3,  'high.school':4, 
        'professional.course':5, 'university.degree':6}
    binary_dict,binary_dict2={'yes':1, 'no':0,'unknown':-1},{'success':1, 'failure':0,'nonexistent':-1}
    contact_dict={'cellular':0, 'telephone':1}
    banking_df['num_loan']=banking_df['loan'].map(binary_dict)
    banking_df['num_housing']=banking_df['housing'].map(binary_dict)
    banking_df['num_default']=banking_df['default'].map(binary_dict)
    banking_df['num_contact']=banking_df['contact'].map(contact_dict)
    banking_df['num_poutcome']=banking_df['poutcome'].map(binary_dict2)
    banking_df['num_education']=banking_df['education'].map(edu_dict)
    banking_df['bin_marital']=0.0
    banking_df.loc[banking_df['marital']=='married','bin_marital']=1.0
    banking_df['Label']=banking_df['y'].map(binary_dict)
    ################################################################
    enc = OrdinalEncoder()
    enc.fit(banking_df[['job']])
    banking_df['num_job']=enc.transform(banking_df[['job']])
    ###########################################################
    all_numeric=numeric+['num_education', 'num_loan','num_housing','num_contact','num_poutcome', 'num_default',
                         'num_job', 'bin_marital']
  #  remaining_ftrs=['job']
    all_features=all_numeric#+remaining_ftrs
    # split into train and test
    X_train, X_test, Y_train, Y_test = train_test_split(banking_df[all_features], banking_df['Label'], test_size=0.20, random_state=42)

    trainset, testset=X_train.copy(),X_test.copy()
    trainset['Label'],testset['Label']=Y_train, Y_test
    dname='banking'
    trainset.to_csv(datapath+'%s/%s_trainset.csv'%(dname,dname), index=False)
    testset.to_csv(datapath+'%s/%s_testset.csv'%(dname,dname), index=False)
else:
    print('Files exist')
    trainset=pd.read_csv(datapath+'%s/%s_trainset.csv'%(dname,dname))
    testset=pd.read_csv(datapath+'%s/%s_testset.csv'%(dname,dname))

print(trainset.shape, testset.shape)
edu_counts=X_train['num_education'].value_counts()

print(edu_dict)

print('Shapes of train and test:', trainset.shape, testset.shape)
print(X_train['bin_marital'].value_counts())
print(X_train['num_housing'].value_counts())
print(X_train['num_loan'].value_counts())
print(X_train['num_contact'].value_counts())
print(X_train['num_poutcome'].value_counts())
print(X_train['num_default'].value_counts())
print(X_train['num_job'].value_counts())
trainset.head(3)

Files do not exist
nan rows: 0
(32950, 19) (8238, 19)
{'illiterate': -1, 'unknown': 0, 'basic.4y': 1, 'basic.6y': 2, 'basic.9y': 3, 'high.school': 4, 'professional.course': 5, 'university.degree': 6}
Shapes of train and test: (32950, 19) (8238, 19)
bin_marital
1.0    19823
0.0    13127
Name: count, dtype: int64
num_housing
 1    17257
 0    14882
-1      811
Name: count, dtype: int64
num_loan
 0    27135
 1     5004
-1      811
Name: count, dtype: int64
num_contact
0    20931
1    12019
Name: count, dtype: int64
num_poutcome
-1    28437
 0     3423
 1     1090
Name: count, dtype: int64
num_default
 0    26090
-1     6857
 1        3
Name: count, dtype: int64
num_job
0.0     8328
1.0     7439
9.0     5352
7.0     3212
4.0     2310
5.0     1363
6.0     1153
2.0     1145
3.0      867
10.0     798
8.0      721
11.0     262
Name: count, dtype: int64


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,num_education,num_loan,num_housing,num_contact,num_poutcome,num_default,num_job,bin_marital,Label
12556,40,94,2,999,0,1.4,93.918,-42.7,4.960,5228.1,3,0,1,1,-1,-1,1.0,1.0,0
35451,31,116,4,999,0,-1.8,92.893,-46.2,1.244,5099.1,6,0,0,0,-1,0,0.0,1.0,0
30592,59,13,6,999,1,-1.8,92.893,-46.2,1.354,5099.1,1,0,0,0,0,0,5.0,1.0,0


### Adult census

In [4]:
adult_datapath=main_data_dir+'/datasets/adult/'
dname='adult'
# retrieve dataset
# categorize attributes
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'label']
if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
    print('Files do not exist')
    train_df = pd.read_csv(adult_datapath+'adult_data', header=None, names=columns, sep=',')
    test_df = pd.read_csv(adult_datapath+'adult.test', header=None, names=columns,sep=',')#all_data=train_df.copy()
    train_df['subset'], test_df['subset']='Train', 'Test'
    all_data=pd.concat([train_df, test_df])
    all_data.rename(columns={'label':'Label'}, inplace=True)
    # fix label columns
    all_data['Label'] = all_data['Label'].apply(lambda x: x.replace('.', ''))
    print('Original shapes train and test df, all_df: ', train_df.shape, test_df.shape, all_data.shape)
    ######################################################################################
    ##Edu hierarchy: levels Preschool < 1st-4th < 5th-6th < 7th-8th < 9th < 10th < 11th
    # < 12th < HS-grad < Prof-school < Assoc-acdm < Assoc-voc < Some-college < Bachelors < Masters < Doctorate .
    edu_dict={'Preschool':1, '1st-4th':2, '5th-6th':3, '7th-8th': 4, '9th':5, '10th':6, '11th':7,
            '12th':8, 'HS-grad':9, 'Prof-school':10, 'Assoc-acdm':11, 'Assoc-voc':12, 'Some-college':13 }
    all_data[all_data=='?'] = np.nan#test_df[test_df == '?'] = np.nan
    ######################################################################
    bin_dict={' Male':1, ' Female':0}
    all_data['num-sex']=all_data['sex'].map(bin_dict) #test_df['num-sex']=test_df['sex'].map(bin_dict)
    #print(train_df['education_num'].value_counts())
    nat_country={
        ' United-States':'Reg_US', ' Cuba':'Reg_LA-Caribbean-Asia', ' Jamaica':'Reg_LA-Caribbean-Asia',
        ' India':'Reg_LA-Caribbean-Asia', ' Mexico':'Reg_LA-Caribbean-Asia',' South':'Reg_LA-Caribbean-Asia',
        ' Puerto-Rico':'Reg_LA-Caribbean-Asia', ' Honduras':'Reg_LA-Caribbean-Asia',' England':'Reg_Europe-CA',
        ' Canada':'Reg_Europe-CA', ' Germany':'Reg_Europe-CA', ' Iran':'Reg_LA-Caribbean-Asia', 
        ' Philippines':'Reg_LA-Caribbean-Asia',' Italy':'Reg_Europe-CA', ' Poland':'Reg_Europe-CA',
        ' Columbia': 'Reg_LA-Caribbean-Asia',' Cambodia':'Reg_LA-Caribbean-Asia', ' Thailand':'Reg_LA-Caribbean-Asia',
        ' Ecuador':'Reg_LA-Caribbean-Asia', ' Laos':'Reg_LA-Caribbean-Asia', ' Taiwan':'Reg_LA-Caribbean-Asia',
        ' Haiti':'Reg_LA-Caribbean-Asia',' Portugal':'Reg_Europe-CA', ' Dominican-Republic':'Reg_LA-Caribbean-Asia',
        ' El-Salvador':'Reg_LA-Caribbean-Asia',' France':'Reg_Europe-CA', ' Guatemala':'Reg_LA-Caribbean-Asia', 
        ' China':'Reg_LA-Caribbean-Asia', ' Japan':'Reg_LA-Caribbean-Asia', ' Yugoslavia':'Reg_Europe-CA',
        ' Peru':'Reg_LA-Caribbean-Asia',' Outlying-US(Guam-USVI-etc)':'Reg_US', ' Scotland':'Reg_Europe-CA',
        ' Trinadad&Tobago':'Reg_LA-Caribbean-Asia',' Greece':'Reg_Europe-CA', ' Nicaragua':'Reg_LA-Caribbean-Asia',
        ' Vietnam':'Reg_LA-Caribbean-Asia', ' Hong': 'Reg_LA-Caribbean-Asia',
        ' Ireland':'Reg_Europe-CA', ' Hungary':'Reg_Europe-CA', ' Holand-Netherlands':'Reg_Europe-CA'}
    all_data['native-region']=all_data['native-country'].map(nat_country)#test_df['native-region']=test_df['native-country'].map(nat_country)
    
    encoded_df0 = pd.get_dummies(all_data['native-region'], columns=['native-region'], dtype=float)
    binary_cats0=list(encoded_df0.columns)
    all_data=pd.concat([all_data, encoded_df0], axis=1)#region_dict={'Reg_US':1, 'Reg_Europe-CA':2, 'Reg_LA-Caribbean-Asia':3}
    #all_data['num-region']=all_data['native-region'].map(region_dict)#test_df['num-region']=test_df['native-region'].map(region_dict)
    ############################################################################################################################
    race_dict={' White':1, ' Black':0.5, ' Asian-Pac-Islander':0.5, ' Amer-Indian-Eskimo':0.5, ' Other':0}
    all_data['race-summary']=all_data['race'].map(race_dict)#test_df['race-summary']=test_df['race'].map(race_dict)
    #####################################################################################################################
    workclass_dict={# any Govt: 1, any
        ' State-gov':'Work_Govt', ' Self-emp-not-inc':'Work_Other', ' Private':'Work_Private', ' Federal-gov':'Work_Govt',
        ' Local-gov':'Work_Govt', ' Self-emp-inc': 'Work_Other', ' Without-pay':'Work_Other', ' Never-worked':'Work_Other'}
    earnclass_dict={# any Govt: 1, any
        ' State-gov':1, ' Self-emp-not-inc':1, ' Private':1, ' Federal-gov':1,
        ' Local-gov':1, ' Self-emp-inc': 1, ' Without-pay':0, ' Never-worked':0}
    all_data['sum-workclass']=all_data['workclass'].map(workclass_dict)#test_df['sum-workclass']=test_df['workclass'].map(workclass_dict)
    all_data['num-earnclass']=all_data['workclass'].map(earnclass_dict)#test_df['num-earnclass']=test_df['workclass'].map(earnclass_dict)
    print(all_data['sum-workclass'].value_counts()/all_data.shape[0])
    print(all_data['num-earnclass'].value_counts()/all_data.shape[0])
    encoded_df1 = pd.get_dummies(all_data['sum-workclass'], columns=['sum-workclass'], dtype=float)
    binary_cats1=list(encoded_df1.columns)
    all_data=pd.concat([all_data, encoded_df1], axis=1)
    print(all_data.shape)
    #####################################################################################################################
    marital_dict={' Married-civ-spouse':'Married',' Married-spouse-absent':'Married',' Married-AF-spouse':'Married', 
                   ' Divorced':'Other', ' Separated':'Other',' Widowed': 'Other', ' Never-married': 'Unmarried'}
    marital_num_dict={'Married':1, 'Unmarried':2, 'Other':0}
    all_data['sum_marital']=all_data['marital-status'].map(marital_dict)#test_df['sum_marital']=test_df['marital-status'].map(marital_dict)
    all_data['num-marital']=all_data['sum_marital'].map(marital_num_dict)#test_df['num-marital']=test_df['sum_marital'].map(marital_num_dict)
    print(all_data['num-marital'].value_counts()/all_data.shape[0])
    #####################################################################################################################
    # remove select columns
    remove_cols = ['education','native-country', 'race', 'workclass', 'sex', 'marital-status','sum_marital',
                  'occupation', 'relationship', 'sum-workclass', 'num-earnclass', 'native-region']
    if len(remove_cols) > 0:       
        all_data = all_data.drop(columns=remove_cols)# test_df = test_df.drop(columns=remove_cols)
        columns = [x for x in columns if x not in remove_cols]
    # Find nan cols
    nancols=all_data.isnull().sum()#/all_data.shape[0]
    # remove nan rows
    all_nan_rows = all_data[all_data.isnull().any(axis=1)]#test_nan_rows = test_df[test_df.isnull().any(axis=1)]
    print('all nan rows: {}'.format(len(all_nan_rows)))#print('test nan rows: {}'.format(len(test_nan_rows)))
    all_data = all_data.dropna()#test_df = test_df.dropna()
    ######################################################################################################################
    numeric = ['age', 'num-sex', 'race-summary', 'education-num', 'num-marital', 'fnlwgt', 'capital-gain',
               'capital-loss', 'hours-per-week']+binary_cats0+binary_cats1
    print('Updated shapes all_df: ', all_data.shape)
    all_data.isnull().sum()/all_data.shape[0]
    #####################################################################################################################
    categorical = list(set(all_data.columns) - set(numeric) - set(label))
    all_data[numeric+['Label','subset']].head(5)
    trainset=all_data[numeric+['Label']].loc[all_data['subset']=='Train'].copy()
    testset=all_data[numeric+['Label']].loc[all_data['subset']=='Test'].copy()
    trainset.to_csv(datapath+'%s/%s_trainset.csv'%(dname,dname), index=False)
    testset.to_csv(datapath+'%s/%s_testset.csv'%(dname,dname), index=False)
else:
    print('Files exist')
    trainset=pd.read_csv(datapath+'%s/%s_trainset.csv'%(dname,dname))
    testset=pd.read_csv(datapath+'%s/%s_testset.csv'%(dname,dname))
    if (trainset['Label'].unique()[0]==' <=50K')| (trainset['Label'].unique()[1]==' >50K'):
        print(trainset['Label'].unique())
        label_dict={' <=50K':0, ' >50K':1}
        trainset['Label']=trainset['Label'].map(label_dict)
        testset['Label']=testset['Label'].map(label_dict)
        print(trainset['Label'].unique())
        trainset.to_csv(datapath+'%s/%s_trainset.csv'%(dname,dname), index=False)
        testset.to_csv(datapath+'%s/%s_testset.csv'%(dname,dname), index=False)
    else:
        print(trainset['Label'].unique())

Files exist
<StringArray>
[' <=50K', ' >50K']
Length: 2, dtype: str
[0 1]


In [5]:
trainset['Label'].value_counts()

Label
0    24720
1     7841
Name: count, dtype: int64

### Surgical

In [33]:
surgical_datapath=main_data_dir+'/datasets/surgical/'
dname='surgical'
if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
    print('Files do not exist')
    surgical_df = pd.read_csv(surgical_datapath+'Surgical-deepnet.csv')
    features=surgical_df.columns[:-1]
    surgical_df.rename(columns={'complication':'Label'}, inplace=True)
    Xtrain, Xtest, Ytrain, Ytest = train_test_split(surgical_df[features], surgical_df['Label'], 
                                                    test_size=0.20, random_state=42)
    trainset, testset=Xtrain.copy(),Xtest.copy()
    trainset['Label'],testset['Label']=Ytrain, Ytest
    trainset.to_csv(datapath+'%s/%s_trainset.csv'%(dname,dname), index=False)
    testset.to_csv(datapath+'%s/%s_testset.csv'%(dname,dname), index=False)
else:
    print('Files exist')
    trainset=pd.read_csv(datapath+'%s/%s_trainset.csv'%(dname,dname))
    testset=pd.read_csv(datapath+'%s/%s_testset.csv'%(dname,dname))

Files do not exist


## Flight delays

In [84]:
type(name_delim)

pandas.Series

In [9]:
dname='flight_delays'
flightdelay_datapath=main_data_dir+'/datasets/%s/'%dname

#if os.path.exists(datapath+'%s/%s_trainset.csv'%(dname,dname))==False:
#print('Files do not exist')
fligh_train_df = pd.read_csv('%s/%s_train.csv'%(flightdelay_datapath, dname))
print(fligh_train_df['Month'].value_counts())
print(fligh_train_df['UniqueCarrier'].value_counts())
print(fligh_train_df['Origin'].value_counts())
fligh_train_df.head(4)

Month
c-8     8830
c-7     8706
c-3     8595
c-5     8543
c-6     8414
c-4     8408
c-10    8405
c-12    8265
c-11    8178
c-9     8163
c-1     8075
c-2     7418
Name: count, dtype: int64
UniqueCarrier
WN    15082
AA     9418
DL     8128
MQ     7443
OO     7390
UA     6876
US     6482
NW     6403
XE     5901
OH     4594
CO     4334
EV     3930
FL     3039
AS     2222
YV     2128
B6     1838
HP     1378
F9     1006
DH      966
HA      762
TZ      446
AQ      234
Name: count, dtype: int64
Origin
ATL    5834
ORD    4870
DFW    4270
LAX    3259
IAH    3048
       ... 
VIS       1
GST       1
WYS       1
ILG       1
VCT       1
Name: count, Length: 289, dtype: int64


,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N


In [21]:
ath_freq=olympics_df['Name'].value_counts()
ath_freq2=olympics_df['Name_delim'].value_counts()
#ath_freq[ath_freq.values>1].keys()

ath_names=np.array(list(ath_freq[ath_freq.values>1].keys()))
ath_names2=np.array(list(ath_freq2[ath_freq2.values>1].keys()))
#ath_names = [item for item in ath_names]
#ath_names = [item for item in ath_names]
regex = fr'^{"|".join(ath_names)}$'
print(len(ath_names), len(ath_names2))
#str_compare=np.stack([ath_names,ath_names2], 1)
#print(str_compare[:10])
ath_names2

57723 57724


array(['Robert Tait McKenzie', 'Heikki Ilmari Savolainen',
       'Joseph  Josy  Stoffel', ..., 'Aleksandr Viktorovich Zyuzin',
       'Piotr ya', 'Tomasz Ireneusz ya'], dtype='<U100')

In [23]:
repeated=olympics_df.loc[olympics_df['Name_delim'].str.contains("|".join(ath_names2),  na=False)]

In [24]:
repeated.shape

(196687, 16)

In [45]:
print(repeated['Name_delim'].nunique(), repeated['Year'].nunique())
names_missing_age=repeated['Name_delim'].loc[repeated['Age'].isnull()].unique()#.to_list()
repeated.loc[repeated['Name_delim'].str.contains(names_missing_age[8], na=False)]

60302 35


,ID,Name_delim,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
564,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 4 x 100 metres Freestyle Relay,NaN
565,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 4 x 200 metres Freestyle Relay,NaN
566,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 100 metres Backstroke,NaN
567,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 200 metres Backstroke,NaN
568,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 200 metres Individual Medley,NaN
569,326,Mohamed Abdullah,Mohamed Abdullah,M,NaN,176.0,66.0,United Arab Emirates,UAE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Men's 4 x 100 metres Medley Relay,NaN
571,328,Mohamed Abdullah,Mohamed Abdullah,M,NaN,180.0,76.0,Iraq,IRQ,1960 Summer,1960,Summer,Roma,Athletics,Athletics Men's Pole Vault,NaN
3450,1959,Mohamed Abdullah Salim Al Houti,Mohamed Abdullah Salim Al-Houti,M,23.0,172.0,69.0,Oman,OMA,1996 Summer,1996,Summer,Atlanta,Athletics,Athletics Men's 200 metres,NaN
3451,1959,Mohamed Abdullah Salim Al Houti,Mohamed Abdullah Salim Al-Houti,M,28.0,172.0,69.0,Oman,OMA,2000 Summer,2000,Summer,Sydney,Athletics,Athletics Men's 200 metres,NaN
3452,1959,Mohamed Abdullah Salim Al Houti,Mohamed Abdullah Salim Al-Houti,M,28.0,172.0,69.0,Oman,OMA,2000 Summer,2000,Summer,Sydney,Athletics,Athletics Men's 4 x 100 metres Relay,NaN


In [37]:
names_missing_age

152        Georgi Abadzhiev
153        Georgi Abadzhiev
212       Sayed Fahmy Abaza
213       Sayed Fahmy Abaza
302         Ismail Abdallah
                ...        
270356      Khristos Zorbas
270357      Khristos Zorbas
270358      Khristos Zorbas
270364      Aleksandar Zori
270365      Aleksandar Zori
Name: Name_delim, Length: 4901, dtype: str